# Pap Smear Cytology Classification — ConvNeXt-Small

**Classes:** NILM (normal) · LSIL (mild abnormality) · HSIL (severe / precancerous)  
**Dataset:** Brown Multicellular ThinPrep Database — 200 images per class (600 total)  
**Split:** 60/20/20 train/val/test — matches the paper exactly (120 test images vs paper's 120)

**Why ConvNeXt-Small over ResNet50/EfficientNet:**
- Modernized CNN architecture (2022) using ViT design principles — depthwise convolutions, larger kernels, LayerNorm, GELU
- Consistently outperforms ResNet50 and EfficientNet on small medical imaging benchmarks
- Retains CNN inductive biases that are advantageous on small datasets like this one

---
### Instructions
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your image folder to Google Drive (flat folder of 600 images)
3. Update `RAW_DATA_PATH` in Cell 2 to point to your Drive folder
4. Fill in the paper's numbers in `PAPER_RESULTS` in Cell 7
5. **Runtime → Run all**

| Cell | What it does |
|------|--------------|
| 1 | Install dependencies |
| 2 | Mount Google Drive + config |
| 3 | Data setup — 60/20/20 stratified split |
| 4 | EDA — class counts, image stats, sample grid |
| 5 | ConvNeXt-Small — model definition + training loop |
| 6 | Evaluation — standard + TTA, confusion matrix, ROC curves |
| 7 | Comparison report vs paper baselines |
| 8 | Download all results as zip |

## Cell 1 — Install dependencies

In [ ]:
!pip install -q torch torchvision torchmetrics scikit-learn matplotlib seaborn tqdm pillow

## Cell 2 — Mount Google Drive + global config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch

# ✏️  Update this to where your image folder lives in Drive
RAW_DATA_PATH = Path('/content/drive/MyDrive/Brown Multicellular ThinPrep Database - images')

DATA_ROOT  = Path('/content/data')
RESULTS    = Path('/content/results/convnext_small')
EDA_OUT    = Path('/content/eda_outputs')
REPORT_OUT = Path('/content/report')

for p in [DATA_ROOT, RESULTS, EDA_OUT, REPORT_OUT]:
    p.mkdir(parents=True, exist_ok=True)

CLASSES     = ['NILM', 'LSIL', 'HSIL']
NUM_CLASSES = len(CLASSES)
SEED        = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device          : {DEVICE}')
print(f'Raw data exists : {RAW_DATA_PATH.exists()}')

## Cell 3 — Data setup (60/20/20 split)
Parses filenames (`HSIL (1).jpg`, `NIL (43).jpg`) and creates a stratified 60/20/20 train/val/test split — matching the paper exactly.  
Result: **120 train · 40 val · 40 test** per class.

In [ ]:
import random
import shutil
from collections import defaultdict

LABEL_MAP  = {'HSIL': 'HSIL', 'LSIL': 'LSIL', 'NIL': 'NILM'}
IMG_EXTS   = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
VAL_RATIO  = 0.20   # 60/20/20 — matches paper
TEST_RATIO = 0.20


def parse_label(filename):
    name = Path(filename).stem
    for raw in LABEL_MAP:
        if name.upper().startswith(raw):
            return LABEL_MAP[raw]
    return None


def build_split(src, dst, seed=42):
    if not src.exists():
        raise FileNotFoundError(f'Source not found: {src}')
    if dst.exists():
        shutil.rmtree(dst)

    by_class = defaultdict(list)
    unrecognized = []
    for f in sorted(src.iterdir()):
        if f.suffix.lower() not in IMG_EXTS:
            continue
        label = parse_label(f.name)
        if label is None: unrecognized.append(f.name)
        else: by_class[label].append(f)

    if unrecognized:
        print(f'⚠  {len(unrecognized)} unrecognized files skipped')

    print('\n── Found images ──────────────────────────────')
    for cls, files in sorted(by_class.items()):
        print(f'  {cls:<6}: {len(files)}')
    print(f'  Total : {sum(len(v) for v in by_class.values())}')

    rng = random.Random(seed)
    splits = {'train': {}, 'val': {}, 'test': {}}

    print('\n── Split sizes (60/20/20) ────────────────────')
    for cls, files in sorted(by_class.items()):
        rng.shuffle(files)
        n       = len(files)
        n_val   = max(1, round(n * VAL_RATIO))
        n_test  = max(1, round(n * TEST_RATIO))
        n_train = n - n_val - n_test
        splits['test'][cls]  = files[:n_test]
        splits['val'][cls]   = files[n_test:n_test+n_val]
        splits['train'][cls] = files[n_test+n_val:]
        print(f'  {cls:<6}: train={n_train}  val={n_val}  test={n_test}')

    total = 0
    for split_name, class_dict in splits.items():
        for cls, files in class_dict.items():
            out = dst / split_name / cls
            out.mkdir(parents=True, exist_ok=True)
            for i, src_f in enumerate(files):
                ext = src_f.suffix.lower()
                shutil.copy2(src_f, out / f'{cls}_{split_name}_{i+1:03d}{ext}')
                total += 1

    print(f'\n✓  {total} images copied to {dst}/')
    print(f'  {"Class":<8}  {"train":<8}  {"val":<8}  {"test":<8}')
    print('  ' + '-'*38)
    for cls in sorted(by_class):
        row = [len(list((dst/s/cls).iterdir())) for s in ['train','val','test']]
        print(f'  {cls:<8}  {row[0]:<8}  {row[1]:<8}  {row[2]:<8}')


build_split(RAW_DATA_PATH, DATA_ROOT, seed=SEED)

## Cell 4 — Exploratory Data Analysis

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display


def count_samples(root):
    counts = {}
    for split in ['train', 'val', 'test']:
        counts[split] = {}
        for cls in CLASSES:
            p = root / split / cls
            imgs = list(p.glob('*.[jp][pn]g')) + list(p.glob('*.jpeg')) if p.exists() else []
            counts[split][cls] = len(imgs)
    return counts


counts = count_samples(DATA_ROOT)

print('='*52)
print('  EDA — Pap Smear Classification')
print('='*52)
for split, cls_counts in counts.items():
    total = sum(cls_counts.values())
    print(f'\n{split.upper()} ({total} images)')
    for cls in CLASSES:
        n = cls_counts[cls]
        print(f'  {cls:<6}: {n:>4}  ({100*n/max(total,1):.1f}%)')

train_vals = [counts['train'][c] for c in CLASSES]
ratio = max(train_vals) / max(min(train_vals), 1)
print(f'\nImbalance ratio (train): {ratio:.2f}x')
print('  → Balanced ✓' if ratio <= 1.5 else '  → Class weights will be applied')

# Image size check
print('\n── Image size sample ─────────────────────────')
widths, heights = [], []
for cls in CLASSES:
    for p in list((DATA_ROOT/'train'/cls).glob('*.[jp][pn]g'))[:10]:
        try:
            with Image.open(p) as im: widths.append(im.width); heights.append(im.height)
        except: pass
if widths:
    print(f'  Width  : {min(widths)} – {max(widths)}  (mean {np.mean(widths):.0f})')
    print(f'  Height : {min(heights)} – {max(heights)}  (mean {np.mean(heights):.0f})')

# Class distribution plot
bar_colors = ['#4C8BB5', '#E07B39', '#5BAD72']
fig, axes  = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(CLASSES)); w = 0.25
for i, (split, color) in enumerate(zip(['train','val','test'], bar_colors)):
    vals = [counts[split][c] for c in CLASSES]
    axes[0].bar(x + i*w, vals, w, label=split, color=color, alpha=0.85)
axes[0].set_xticks(x+w); axes[0].set_xticklabels(CLASSES, fontsize=12)
axes[0].set_ylabel('Images'); axes[0].set_title('Class distribution per split')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
tv = [counts['train'][c] for c in CLASSES]; tot = sum(tv)
axes[1].pie(tv, labels=[f'{c}\n({v} | {v/tot*100:.1f}%)' for c,v in zip(CLASSES,tv)],
            colors=bar_colors, startangle=90)
axes[1].set_title('Train set class balance')
plt.tight_layout()
plt.savefig(EDA_OUT/'class_distribution.png', dpi=150, bbox_inches='tight')
display(plt.gcf()); plt.close()

# Sample grid
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(12, 9))
fig.suptitle('Sample images per class (train)', fontsize=13, y=1.01)
for row, cls in enumerate(CLASSES):
    imgs = list((DATA_ROOT/'train'/cls).glob('*.[jp][pn]g'))[:4]
    for col in range(4):
        ax = axes[row][col]; ax.axis('off')
        if col < len(imgs):
            try: ax.imshow(Image.open(imgs[col]).convert('RGB'))
            except: pass
        if col == 0: ax.set_title(cls, fontsize=11, loc='left')
plt.tight_layout()
plt.savefig(EDA_OUT/'sample_grid.png', dpi=150, bbox_inches='tight')
display(plt.gcf()); plt.close()
print(f'EDA plots saved to {EDA_OUT}/')

## Cell 5 — ConvNeXt-Small training

**Architecture:** ConvNeXt-Small pretrained on ImageNet-1K  
**Stage 1 (epochs 1–8):** Only the classifier head is trained (backbone frozen)  
**Stage 2 (epochs 9–60):** Full model fine-tuned at a lower learning rate  
**Key extras:** Mixup augmentation · label smoothing · cosine LR · weighted sampler · gradient clipping

In [ ]:
import json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from torchmetrics import Accuracy, F1Score, AUROC
from tqdm.notebook import tqdm

torch.manual_seed(SEED); np.random.seed(SEED)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE       = 224
BATCH_SIZE     = 32
NUM_EPOCHS     = 70
LR_HEAD        = 3e-4      # phase 1: head only
LR_BACKBONE    = 1e-5      # phase 2: backbone (10x lower than before — fixes nan)
LR_HEAD_FT     = 5e-5      # phase 2: head during fine-tune
WARMUP_EPOCHS  = 3         # linear warmup after unfreeze before cosine kicks in
UNFREEZE_EPOCH = 8
LABEL_SMOOTH   = 0.1
MIXUP_ALPHA    = 0.3
GRAD_CLIP      = 0.5       # tighter clipping during unfreeze phase
TTA_RUNS       = 5
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── Transforms ────────────────────────────────────────────────────────────────
# RandomErasing and ElasticTransform help LSIL which sits between normal/precancerous
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE+32, IMG_SIZE+32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ElasticTransform(alpha=50.0, sigma=5.0),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
tta_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE+16, IMG_SIZE+16)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Mixup ─────────────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.3):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam*criterion(pred, y_a) + (1-lam)*criterion(pred, y_b)

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = datasets.ImageFolder(DATA_ROOT/'train', transform=train_tf)
val_ds   = datasets.ImageFolder(DATA_ROOT/'val',   transform=val_tf)
test_ds  = datasets.ImageFolder(DATA_ROOT/'test',  transform=val_tf)

labels       = [s[1] for s in train_ds.samples]
class_counts = np.bincount(labels)
sample_wts   = 1.0 / class_counts[labels]
sampler      = WeightedRandomSampler(sample_wts, len(sample_wts), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True)

loss_weights = torch.tensor(1.0/class_counts, dtype=torch.float).to(DEVICE)
loss_weights = loss_weights / loss_weights.sum() * NUM_CLASSES

print(f'Class mapping  : {train_ds.class_to_idx}')
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')
print(f'Samples per class (train): {class_counts}')

# ── Model ─────────────────────────────────────────────────────────────────────
weights = models.ConvNeXt_Small_Weights.IMAGENET1K_V1
model   = models.convnext_small(weights=weights)
in_feat = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_feat, NUM_CLASSES)
for name, p in model.named_parameters():
    if 'classifier' not in name:
        p.requires_grad = False
model = model.to(DEVICE)
total  = sum(p.numel() for p in model.parameters())
active = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total:,} total  |  {active:,} trainable (head only)')

# ── Optimizer / scheduler ─────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCH)
scaler    = torch.amp.GradScaler(enabled=DEVICE.type=='cuda')

# ── Training helpers ──────────────────────────────────────────────────────────
def train_one_epoch(model, loader):
    model.train()
    loss_sum = correct = total = 0
    for imgs, lbls in tqdm(loader, desc='  train', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        imgs, y_a, y_b, lam = mixup_data(imgs, lbls, MIXUP_ALPHA)
        optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                            enabled=DEVICE.type=='cuda'):
            logits = model(imgs)
            loss   = mixup_criterion(criterion, logits, y_a, y_b, lam)
        if torch.isnan(loss):
            print('  ⚠  NaN loss detected — skipping batch')
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item()*lbls.size(0)
        correct  += (logits.argmax(1)==lbls).sum().item()
        total    += lbls.size(0)
    return (loss_sum/total if total else float('nan')), (correct/total if total else 0)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    all_probs, all_preds, all_lbls = [], [], []
    for imgs, lbls in tqdm(loader, desc='  eval', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, lbls)
        probs  = torch.softmax(logits, 1)
        preds  = logits.argmax(1)
        loss_sum += loss.item()*lbls.size(0)
        correct  += (preds==lbls).sum().item()
        total    += lbls.size(0)
        all_probs.append(probs.cpu()); all_preds.append(preds.cpu()); all_lbls.append(lbls.cpu())
    return loss_sum/total, correct/total, torch.cat(all_probs), torch.cat(all_preds), torch.cat(all_lbls)


# ── Training loop ─────────────────────────────────────────────────────────────
best_val_f1 = 0.0
history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
unfrozen = False

print(f'\n{"="*60}')
print(f'  Training ConvNeXt-Small  ({NUM_EPOCHS} epochs)')
print(f'{"="*60}')

for epoch in range(1, NUM_EPOCHS+1):

    # ── Phase 2: discriminative LRs — backbone much lower than head ───────────
    if epoch == UNFREEZE_EPOCH and not unfrozen:
        backbone_params = [p for n,p in model.named_parameters() if 'classifier' not in n]
        head_params     = [p for n,p in model.named_parameters() if 'classifier' in n]
        optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': LR_BACKBONE},
            {'params': head_params,     'lr': LR_HEAD_FT},
        ], weight_decay=1e-4)
        for p in model.parameters(): p.requires_grad = True
        def lr_lambda(ep):
            if ep < WARMUP_EPOCHS: return (ep + 1) / WARMUP_EPOCHS
            progress = (ep - WARMUP_EPOCHS) / max(NUM_EPOCHS - UNFREEZE_EPOCH - WARMUP_EPOCHS, 1)
            return 0.5 * (1 + np.cos(np.pi * progress))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
        unfrozen = True
        print(f'  → Backbone unfrozen at epoch {epoch}')
        print(f'    backbone LR={LR_BACKBONE}  head LR={LR_HEAD_FT}  ({WARMUP_EPOCHS}-ep warmup)')

    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    vl_loss, vl_acc, vl_probs, vl_preds, vl_lbls = evaluate(model, val_loader)
    scheduler.step()

    if np.isnan(tr_loss) or np.isnan(vl_loss):
        print(f'Ep {epoch:02d}/{NUM_EPOCHS}  ⚠  NaN detected — skipping epoch')
        continue

    vl_f1 = F1Score(task='multiclass', num_classes=NUM_CLASSES, average='macro')(
        vl_preds, vl_lbls).item()

    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss);   history['val_acc'].append(vl_acc)

    star = ''
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), RESULTS/'best_model.pth')
        star = '  ★ best'

    print(f'Ep {epoch:02d}/{NUM_EPOCHS}  '
          f'tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.3f}  '
          f'vl_loss={vl_loss:.4f}  vl_acc={vl_acc:.3f}  '
          f'vl_f1={vl_f1:.4f}  ({time.time()-t0:.0f}s){star}')

print(f'\nBest val F1: {best_val_f1:.4f}  — checkpoint saved')

## Cell 6 — Evaluation (standard + TTA + ROC curves)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc as sk_auc
from sklearn.preprocessing import label_binarize
from IPython.display import display

model.load_state_dict(torch.load(RESULTS/'best_model.pth', map_location=DEVICE))


def compute_metrics(probs, preds, labels, tag='test'):
    acc      = Accuracy(task='multiclass', num_classes=NUM_CLASSES)(preds, labels).item()
    f1_macro = F1Score(task='multiclass',  num_classes=NUM_CLASSES, average='macro')(preds, labels).item()
    f1_per   = F1Score(task='multiclass',  num_classes=NUM_CLASSES, average='none')(preds, labels).tolist()
    auc      = AUROC(task='multiclass',    num_classes=NUM_CLASSES)(probs, labels).item()
    cm       = confusion_matrix(labels.numpy(), preds.numpy())
    report   = classification_report(labels.numpy(), preds.numpy(), target_names=CLASSES, digits=4)
    print(f'\n{"-"*52}  [{tag.upper()}]')
    print(f'  Accuracy   : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Macro F1   : {f1_macro:.4f}')
    print(f'  AUC-ROC    : {auc:.4f}')
    for cls, s in zip(CLASSES, f1_per): print(f'  F1 {cls:<6} : {s:.4f}')
    print(f'\n{report}')
    return {'accuracy': round(acc,4), 'macro_f1': round(f1_macro,4), 'auc': round(auc,4),
            'f1_per_class': {c: round(s,4) for c,s in zip(CLASSES, f1_per)},
            'confusion_matrix': cm.tolist()}


@torch.no_grad()
def evaluate_tta(model, dataset, n_runs=TTA_RUNS):
    model.eval()
    orig_tf = dataset.transform; dataset.transform = tta_tf
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
    probs_sum = None; all_lbls = []
    for run in range(n_runs):
        rp, rl = [], []
        for imgs, lbls in tqdm(loader, desc=f'  TTA {run+1}/{n_runs}', leave=False):
            rp.append(torch.softmax(model(imgs.to(DEVICE)),1).cpu())
            if run == 0: rl.append(lbls)
        bp = torch.cat(rp)
        probs_sum = bp if probs_sum is None else probs_sum + bp
        if run == 0: all_lbls = torch.cat(rl)
    dataset.transform = orig_tf
    avg = probs_sum / n_runs
    return avg, avg.argmax(1), all_lbls


def plot_cm(cm_arr, title, path):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted label', fontsize=11)
    ax.set_ylabel('True label',      fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    display(plt.gcf()); plt.close()


# Standard eval
_, _, ts_probs, ts_preds, ts_lbls = evaluate(model, test_loader)
metrics_std = compute_metrics(ts_probs, ts_preds, ts_lbls, 'test (standard)')

# TTA eval
tta_probs, tta_preds, tta_lbls = evaluate_tta(model, test_ds)
metrics_tta = compute_metrics(tta_probs, tta_preds, tta_lbls, 'test (TTA)')

# Save raw probs
np.save(RESULTS/'test_probs.npy',  ts_probs.numpy())
np.save(RESULTS/'test_labels.npy', ts_lbls.numpy())

# Confusion matrices
plot_cm(np.array(metrics_std['confusion_matrix']),
        'ConvNeXt-Small — Confusion matrix (standard)', RESULTS/'confusion_matrix.png')
plot_cm(np.array(metrics_tta['confusion_matrix']),
        'ConvNeXt-Small — Confusion matrix (TTA)', RESULTS/'confusion_matrix_tta.png')

# Training history
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, key, title in zip(axes, ['loss','acc'], ['Loss','Accuracy']):
    ax.plot(history[f'train_{key}'], label='train', linewidth=1.5)
    ax.plot(history[f'val_{key}'],   label='val',   linewidth=1.5)
    ax.axvline(UNFREEZE_EPOCH-1, color='gray', linestyle='--', linewidth=1,
               label=f'unfreeze (ep {UNFREEZE_EPOCH})')
    ax.set_title(title, fontsize=12); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('ConvNeXt-Small — Training history', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS/'training_history.png', dpi=150, bbox_inches='tight')
display(plt.gcf()); plt.close()

# ROC curves
labels_bin = label_binarize(ts_lbls.numpy(), classes=range(NUM_CLASSES))
fig, ax    = plt.subplots(figsize=(7, 6))
for i, (cls, color) in enumerate(zip(CLASSES, ['#4C9BE8','#F5A623','#E85D5D'])):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], ts_probs.numpy()[:, i])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{cls}  (AUC = {sk_auc(fpr,tpr):.4f})')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random')
ax.set_xlabel('False positive rate', fontsize=11)
ax.set_ylabel('True positive rate',  fontsize=11)
ax.set_title('ROC curves — ConvNeXt-Small (one-vs-rest)', fontsize=12, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS/'roc_curves.png', dpi=150, bbox_inches='tight')
display(plt.gcf()); plt.close()

# Save metrics JSON
def strip_cm(d): return {k:v for k,v in d.items() if k!='confusion_matrix'}
with open(RESULTS/'metrics.json','w') as f:
    json.dump({'standard': strip_cm(metrics_std), 'tta': strip_cm(metrics_tta)}, f, indent=2)

print(f'\n{"="*60}')
print('  FINAL RESULTS')
print(f'{"="*60}')
for tag, m in [('Standard', metrics_std), ('With TTA ', metrics_tta)]:
    print(f'  {tag}  |  Acc: {m["accuracy"]:.4f}  F1: {m["macro_f1"]:.4f}  AUC: {m["auc"]:.4f}')
print(f'  Paper ResNet50 baseline: Acc: 0.7416')

## Cell 7 — Comparison report vs paper baselines
**Fill in `PAPER_RESULTS` below before running.**

In [ ]:
# ✏️  Fill in the paper's reported numbers here
PAPER_RESULTS = {
    'VGG19': {
        'accuracy': 0.6416,
        'macro_f1': 0.000,   # fill in if reported
        'auc':      0.000,
        'f1_per_class': {'NILM': 0.000, 'LSIL': 0.000, 'HSIL': 0.000},
    },
    'ResNet50': {
        'accuracy': 0.7416,
        'macro_f1': 0.000,
        'auc':      0.000,
        'f1_per_class': {'NILM': 0.000, 'LSIL': 0.000, 'HSIL': 0.000},
    },
}

table = dict(PAPER_RESULTS)
table['ConvNeXt-Small']       = metrics_std
table['ConvNeXt-Small (TTA)'] = metrics_tta
models_list = list(table.keys())
palette = ['#4C9BE8', '#F5A623', '#E85D5D', '#6DBF67', '#B07FE0']

# Comparison table
headers = ['Model', 'Accuracy', 'Macro F1', 'AUC-ROC'] + [f'F1 {c}' for c in CLASSES]
rows = []
for m in models_list:
    d = table[m]
    row = [m, f"{d['accuracy']:.4f}", f"{d['macro_f1']:.4f}", f"{d['auc']:.4f}"]
    for c in CLASSES: row.append(f"{d['f1_per_class'].get(c,0):.4f}")
    rows.append(row)

fig, ax = plt.subplots(figsize=(14, 2.2 + 0.55*len(models_list)))
ax.axis('off')
tbl = ax.table(cellText=rows, colLabels=headers, cellLoc='center', loc='center',
               colWidths=[0.22]+[0.12]*(len(headers)-1))
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.6)
for j in range(len(headers)):
    tbl[0,j].set_facecolor('#2B4C7E'); tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(models_list)+1):
    for j in range(len(headers)):
        if 'ConvNeXt' in models_list[i-1]: tbl[i,j].set_facecolor('#EBF5FB')
        elif i%2==0: tbl[i,j].set_facecolor('#F8F9FA')
for col_idx, key in enumerate(['accuracy','macro_f1','auc']):
    vals = [table[m][key] for m in models_list]
    tbl[int(np.argmax(vals))+1, col_idx+1].set_text_props(fontweight='bold', color='#1A5276')
plt.title('Model comparison — ConvNeXt-Small vs paper baselines',
          fontsize=12, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig(REPORT_OUT/'comparison_table.png', dpi=150, bbox_inches='tight')
display(plt.gcf()); plt.close()

# Bar chart
x = np.arange(3); w = 0.8/len(models_list)
fig, ax = plt.subplots(figsize=(10, 5))
for i, (m, color) in enumerate(zip(models_list, palette)):
    vals = [table[m]['accuracy'], table[m]['macro_f1'], table[m]['auc']]
    bars = ax.bar(x+i*w-(len(models_list)-1)*w/2, vals, w*0.9, label=m, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(['Accuracy','Macro F1','AUC-ROC'], fontsize=11)
ax.set_ylim(0, 1.10); ax.set_ylabel('Score')
ax.set_title('Model performance comparison', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_OUT/'comparison_bar.png', dpi=150, bbox_inches='tight')
display(plt.gcf()); plt.close()

# Per-class F1
x = np.arange(len(CLASSES))
fig, ax = plt.subplots(figsize=(9, 5))
for i, (m, color) in enumerate(zip(models_list, palette)):
    vals = [table[m]['f1_per_class'].get(c,0) for c in CLASSES]
    bars = ax.bar(x+i*w-(len(models_list)-1)*w/2, vals, w*0.9, label=m, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(CLASSES, fontsize=11)
ax.set_ylim(0, 1.10); ax.set_ylabel('F1 score')
ax.set_title('Per-class F1 score comparison', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_OUT/'per_class_f1.png', dpi=150, bbox_inches='tight')
display(plt.gcf()); plt.close()

# Text summary
print(f'\n{"="*60}'); print('  FINAL COMPARISON'); print(f'{"="*60}')
print(f'  {"Model":<28} {"Acc":>8} {"F1":>8} {"AUC":>8}'); print('  '+'-'*52)
for m, d in table.items():
    marker = '  ◀ our model' if 'ConvNeXt' in m else ''
    print(f'  {m:<28} {d["accuracy"]:>8.4f} {d["macro_f1"]:>8.4f} {d["auc"]:>8.4f}{marker}')
our_f1   = metrics_tta['macro_f1']
paper_f1 = max(PAPER_RESULTS[m]['macro_f1'] for m in PAPER_RESULTS)
delta    = our_f1 - paper_f1
print(f'\n  ConvNeXt-Small (TTA) macro F1 : {our_f1:.4f}')
print(f'  Best paper baseline macro F1  : {paper_f1:.4f}')
print(f'  {"Improvement : +" if delta>0 else "Gap : "}{abs(delta):.4f}  {"✓ beats baseline" if delta>0 else ""}')
print(f'\n  Report saved to {REPORT_OUT}/')

## Cell 8 — Download all results as zip

In [ ]:
import zipfile
from google.colab import files

zip_path = '/content/pap_smear_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [RESULTS, REPORT_OUT, EDA_OUT]:
        for fpath in folder.rglob('*'):
            if fpath.is_file():
                zf.write(fpath, fpath.relative_to('/content'))

print(f'Zip created: {zip_path}')
files.download(zip_path)